# Initialization

In [9]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
os.environ["USER_AGENT"] = "retrival_agent"

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Tools

In [ ]:
#Data importation 
doc_path = "./datasets/tunisia.txt"
loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()

#First tool
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=1)
texts = text_splitter.split_documents(documents)

Created a chunk of size 1347, which is longer than the specified 1000
Created a chunk of size 2031, which is longer than the specified 1000
Created a chunk of size 1479, which is longer than the specified 1000
Created a chunk of size 2087, which is longer than the specified 1000
Created a chunk of size 1100, which is longer than the specified 1000
Created a chunk of size 1186, which is longer than the specified 1000
Created a chunk of size 1138, which is longer than the specified 1000
Created a chunk of size 1101, which is longer than the specified 1000
Created a chunk of size 1527, which is longer than the specified 1000
Created a chunk of size 1294, which is longer than the specified 1000
Created a chunk of size 1195, which is longer than the specified 1000
Created a chunk of size 1174, which is longer than the specified 1000
Created a chunk of size 1150, which is longer than the specified 1000
Created a chunk of size 1232, which is longer than the specified 1000
Created a chunk of s

In [12]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings

In [14]:
#Data embedding
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
docsearch = Chroma.from_documents(texts, embeddings, collection_name="tunisian_history")

In [ ]:
from langchain.chains import RetrievalQA

In [19]:
tunisian_history = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [56]:
#Second tool 
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Wikipedia:FAQ/Readers")

In [57]:
docs = loader.load()
wiki_texts = text_splitter.split_documents(docs)
wiki_db = Chroma.from_documents(wiki_texts, embeddings, collection_name="wiki")

wiki = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=wiki_db.as_retriever()
)

Created a chunk of size 1832, which is longer than the specified 1000
Created a chunk of size 1292, which is longer than the specified 1000
Created a chunk of size 3365, which is longer than the specified 1000
Created a chunk of size 1833, which is longer than the specified 1000


# Agent creation

In [58]:
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

In [59]:
tools = [
    Tool(
        name="Tunisa history",
        func=tunisian_history.run,
        description="useful for when you need to answer questions about Tunisia history. Input should be a fully formed question.",
    ),
    Tool(
        name="Wikipedia QA System",
        func=wiki.run,
        description="useful for when you need to answer questions about Wikipedia. Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [60]:
#Inizialise agent 
agent = initialize_agent(
    tools, llm, 
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
    handle_parsing_errors=True, 
    verbose=True
)

In [61]:
agent.invoke(
    "Who was the first Tunisian president?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


To answer the question about the first Tunisian president, I will need to gather historical information specifically about Tunisia. 

Action: Tunisa history  
Action Input: Who was the first Tunisian president?  
Observation: The first Tunisian president was Habib Bourguiba. He served as president from 1957 until 1987.
Thought:I now know the final answer.  
Final Answer: The first Tunisian president was Habib Bourguiba, who served from 1957 until 1987.

> Finished chain.


{'input': 'Who was the first Tunisian president?',
 'output': 'The first Tunisian president was Habib Bourguiba, who served from 1957 until 1987.'}

In [62]:
agent.invoke(
    "Who is legit of German visa?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


To answer the question about who is eligible for a German visa, I should look for information that outlines the criteria and requirements for obtaining a German visa. This information is likely available on Wikipedia or similar resources. 

Action: Wikipedia QA System  
Action Input: Who is eligible for a German visa?  
Observation: I don't know.


> Finished chain.


{'input': 'Who is legit of German visa?', 'output': "I don't know."}

In [63]:
agent.invoke(
    "Is Wikipedia censored?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to determine whether Wikipedia is subject to censorship and what the general consensus or evidence is regarding this issue. This may involve looking for information on Wikipedia's policies or any controversies surrounding censorship. 

Action: Wikipedia QA System  
Action Input: Is Wikipedia censored?  
Observation: Wikipedia itself is not censored, as its policy is to not censor content. The community of Wikipedia editors determines what text and images are displayed, which means there may be material that offends some people. However, access to Wikipedia may be censored by the governments of various countries, and in such cases, using a VPN might be required to access it.


> Finished chain.


{'input': 'Is Wikipedia censored?',
 'output': 'Wikipedia itself is not censored, as its policy is to not censor content. The community of Wikipedia editors determines what text and images are displayed, which means there may be material that offends some people. However, access to Wikipedia may be censored by the governments of various countries, and in such cases, using a VPN might be required to access it.'}